In [1]:
import json

# Open and read the JSON file
with open('/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/training.json', 'r') as file:
    train_data = json.load(file)
with open('/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/test.json', 'r') as file:
    test_data = json.load(file)

In [2]:
for i in train_data:
    if train_data[i]['labels_task3_1'].count('YES') > train_data[i]['labels_task3_1'].count('NO'):
        train_data[i]['labels_task3_1'] = 1
    else:
        train_data[i]['labels_task3_1'] = 0

In [3]:
import pandas as pd
import numpy as np

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)
train_df = train_df.transpose()
test_df = test_df.transpose()

In [4]:
import re
import string

def remove_username(text):
    user = re.compile(r"@[^\s]+")
    return user.sub(r"", text)

def remove_URL(text):
    url = re.compile(r"https?://\S+|www\.\S+")
    return url.sub(r"", text)

def remove_punct(text):
    translator = str.maketrans("", "", string.punctuation)
    return text.translate(translator)

def remove_emoji(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           u"\U0001F923"
                           u"\U0001F972"
                           u"\U0001F973"
                           u"\U0001F9D0"
                           u"\U0001F9E1"
                           u"\U0001F49B"
                           u"\U0001F924"
                           u"\U0001F921"
                           u"\U0001F49C"
                           u"\U0001F914"
                           u"\U0001F97A"
                           u"\U0001F970"
                           u"\U0001F974"
                           u"\U0001F922"
                           u"\U0001F91D"
                           u"\U0001F92A"
                           u"\U0001F92F"
                           u"\U0001F98A"
                           u"\U0001F981"
                           u"\U0001F926"
                           u"\U0001F937"
                           u"\U0001F92B"
                           u"\U0001F92E"
                           u"\U0001F90F"
                           u"\U0001F91F"
                           u"\U0001F92D"
                           u"\U0001F971"
                           u"\U0001F911"
                           u"\U0001F917"
                           u"\U0001F918"
                           u"\U0001F929"
                           u"\U0001F975"
                           u"\U0001F915"
                           u"\U0001F920"
                           u"\U0001F932"
                           u"\U0001F91E"
                           u"\U0001F928"
                           u"\U0001F92C"
                           u"\U0001F976"
                           u"\U0001F9E0"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

In [5]:
train_df['text'] = train_df.text.map(remove_username)
train_df['text'] = train_df.text.map(remove_URL)
train_df['text'] = train_df.text.map(remove_punct)
train_df['text'] = train_df.text.map(remove_emoji)
test_df['text'] = test_df.text.map(remove_username)
test_df['text'] = test_df.text.map(remove_URL)
test_df['text'] = test_df.text.map(remove_punct)
test_df['text'] = test_df.text.map(remove_emoji)

In [6]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop = set(stopwords.words(fileids=('english', 'spanish')))
def remove_stopwords(text):
    filtered_words = [word.lower() for word in text.split() if word.lower() not in stop]
    return " ".join(filtered_words)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/naranja/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
train_df['text'] = train_df.text.map(remove_stopwords)
test_df['text'] = test_df.text.map(remove_stopwords)

In [8]:
csv_file_train = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/training.csv'
csv_file_test = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/test.csv'
train_df.to_csv(csv_file_train, index=False)
test_df.to_csv(csv_file_test, index=False)

In [9]:
train_df.shape

(2006, 14)

In [10]:
csv_file_train_es = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/training_es.csv'
train_df_es = train_df
train_df_es = train_df_es[train_df['lang'] == 'es']
train_df_es.to_csv(csv_file_train_es, index=False)

In [11]:
train_df_es.shape

(1212, 14)

In [12]:
csv_file_train_en = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/training_en.csv'
train_df_en = train_df
train_df_en = train_df_en[train_df['lang'] == 'en']
train_df_en.to_csv(csv_file_train_en, index=False)

In [13]:
train_df_en.shape

(794, 14)

In [14]:
from sklearn.model_selection import train_test_split

# Use train_test_split to split training data into training and validation sets
train_text_es, val_text_es, train_labels_es, val_labels_es, train_id_es, val_id_es = train_test_split(train_df_es["text"].to_numpy(),
                                                                  train_df_es["labels_task3_1"].to_numpy(),
                                                                  train_df_es["id_EXIST"].to_numpy(),
                                                                  test_size=0.1, # dedicate 10% of samples to validation set
                                                                  random_state=42) # random state for reproducibility

In [15]:
len(train_text_es), len(train_labels_es), len(val_text_es), len(val_labels_es)

(1090, 1090, 122, 122)

In [16]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization # after TensorFlow 2.6

# Use the default TextVectorization variables
text_vectorizer = TextVectorization(max_tokens=None, # how many words in the vocabulary (all of the different words in your text)
                                    standardize="lower_and_strip_punctuation", # how to process text
                                    split="whitespace", # how to split tokens
                                    ngrams=None, # create groups of n-words?
                                    output_mode="int", # how to map tokens to numbers
                                    output_sequence_length=None) # how long should the output sequence of tokens be?
                                    # pad_to_max_tokens=True) # Not valid if using max_tokens=None

2025-04-14 21:40:06.153373: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [17]:
# Find average number of tokens (words) in training Tweets
round(sum([len(i.split()) for i in train_text_es])/len(train_text_es))

55

In [18]:
# Setup text vectorization with custom variables
max_vocab_length = 10000 # max number of words to have in our vocabulary
max_length = 12 # max length our sequences will be (e.g. how many words from a Tweet does our model see?)

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [19]:
# Fit the text vectorizer to the training text
text_vectorizer.adapt(train_text_es)

In [20]:
# Choose a random sentence from the training dataset and tokenize it
import random
random_sentence = random.choice(train_text_es)
print(f"Original text:\n{random_sentence}\
      \n\nVectorized version:")
text_vectorizer([random_sentence])

Original text:
misma describí asombré facescreaminginfear womanwithwhitecanemediumlightskintone camerawithflash p cyndidelrio mujer cabello negro sombrero 23 años posiblemente pelo rizado hola ¿ sabías hecho ser ciega impide conocer imágenes gracias aplicaciones tecnología mira aquí foto preparada entonces voy ir aquí dice compartir voy ir cada municipiosakia reconociendo imageny puedo enterar contenido texto memes dale like follow si conociste información      

Vectorized version:


<tf.Tensor: shape=(1, 12), dtype=int64, numpy=
array([[ 333,    1,    1, 1116, 6420, 3430,  526,    1,   12,  514,  714,
        7647]])>

In [21]:
# Get the unique words in the vocabulary
words_in_vocab = text_vectorizer.get_vocabulary()
top_5_words = words_in_vocab[:5] # most common tokens (notice the [UNK] token for "unknown" words)
bottom_5_words = words_in_vocab[-5:] # least common tokens
print(f"Number of words in vocab: {len(words_in_vocab)}")
print(f"Top 5 most common words: {top_5_words}") 
print(f"Bottom 5 least common words: {bottom_5_words}")

Number of words in vocab: 10000
Top 5 most common words: ['', '[UNK]', '¡', 'catedral', 'oh']
Bottom 5 least common words: ['motar', 'motais', 'mot', 'mostró', 'mostrarme']


In [22]:
tf.random.set_seed(42)
from tensorflow.keras import layers

embedding = layers.Embedding(input_dim=max_vocab_length, # set input shape
                             output_dim=128, # set size of embedding vector
                             embeddings_initializer="uniform", # default, intialize randomly
                             input_length=max_length, # how long is each input
                             name="embedding_1")
embedding

/Users/naranja/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


<Embedding name=embedding_1, built=False>

In [23]:
# Get a random sentence from training set
result = random.choice(train_text_es)
print(f"Original text:\n{result}\
      \n\nEmbedded version:")

# Embed the random sentence (turn it into numerical representation)
sample_embed = embedding(text_vectorizer([result]))
sample_embed

Original text:
policecarlight manpoliceofficermediumskintone ¡2 bandas desarticuladas azuay conoce backhandindexpointingrightmediumskintone abad resultados feriado nueve octubre ocupación hotelera superior 75 operativos control carreteras desarticulación 2 bandas clausura eventos locales permisos feriado nivel azuario obtenido ocupación hotelera superior setenta cinco cientoa nivel cuenca desarrollan importante operativo control carreteras anticipado comprometido ciudadaníaen resultadas dos bandas organizadas sido desarticuladas intendencia clausurado eventos locales comerciales provincia contado permisos necesarios      

Embedded version:


<tf.Tensor: shape=(1, 12, 128), dtype=float32, numpy=
array([[[ 0.02044174,  0.01164713,  0.00535532, ..., -0.03197279,
         -0.02935027, -0.04349132],
        [ 0.04073015,  0.00151347, -0.00368323, ...,  0.00082753,
          0.02984792, -0.03926746],
        [ 0.01200002, -0.02871316,  0.00734764, ...,  0.03562225,
          0.0388914 ,  0.04735613],
        ...,
        [ 0.03218361, -0.01859555, -0.00888175, ..., -0.01689357,
          0.00461191, -0.01092001],
        [ 0.04572051, -0.02919195,  0.00646299, ..., -0.04937657,
          0.04256486,  0.04623132],
        [-0.02368172,  0.04111587, -0.00378928, ...,  0.0489636 ,
          0.03910759, -0.04120042]]], dtype=float32)>

In [24]:
# Model 0: Getting a baseline - Naive Bayes
train_labels_es = train_labels_es.astype('int') # sklearn cannot recognize object type
val_labels_es = val_labels_es.astype('int')

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Create tokenization and modelling pipeline
model_0 = Pipeline([
                    ("tfidf", TfidfVectorizer()), # convert words to numbers using tfidf
                    ("clf", MultinomialNB()) # model the text
])

# Fit the pipeline to the training data
model_0.fit(train_text_es, train_labels_es)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [26]:
baseline_score = model_0.score(val_text_es, val_labels_es)
print(f"Our baseline model achieves an accuracy of: {baseline_score*100:.2f}%")

Our baseline model achieves an accuracy of: 65.57%


In [27]:
# Make predictions
baseline_preds = model_0.predict(val_text_es)
baseline_preds[:20],

(array([0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1]),)

In [28]:
# Prediction on test dataset
test_sentences = test_df["text"].to_list()
for test_sample in test_sentences:
    pred_prob = tf.squeeze(model_0.predict([test_sample])) # has to be list
    pred = tf.round(pred_prob)
    print(f"Pred: {int(pred)}")

Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 0


In [29]:
baseline_preds = list(map(lambda x: "YES" if x == 1 else "NO", baseline_preds))
baseline_preds = np.array(baseline_preds)
val_labels_arr = list(map(lambda x: "YES" if x == 1 else "NO", val_labels_es))
val_labels_arr = np.array(val_labels_arr)

In [30]:
import pandas as p

preds_dict = {
    'test_case': ['EXIST2025']*len(val_text_es),
    'id': val_id_es,
    'value': baseline_preds,
}

preds_dict = pd.DataFrame(preds_dict)
output_path = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/predictions_es.json'
with open(output_path, 'w', encoding='utf-8') as output_file:
       output_file.write(preds_dict.to_json(orient='records'))

In [31]:
val_dict = {
    'test_case': ['EXIST2025']*len(val_text_es),
    'id': val_id_es,
    'value': val_labels_arr,
}

val_dict = pd.DataFrame(val_dict)
output_path = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/goldgroundtruth_es.json'
with open(output_path, 'w', encoding='utf-8') as output_file:
       output_file.write(val_dict.to_json(orient='records'))

In [32]:
from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils
from pyevall.metrics.metricfactory import MetricFactory

test = PyEvALLEvaluation()
preds = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/predictions_es.json'
labels = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/goldgroundtruth_es.json'
metrics = [
    MetricFactory.Accuracy.value,
    MetricFactory.FMeasure.value,
]
params= dict()
report = test.evaluate(preds, labels, metrics, **params) 
report.print_report()

2025-04-14 21:40:10,974 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['Accuracy', 'FMeasure']
2025-04-14 21:40:11,019 - pyevall.metrics.metrics - INFO -             evaluate() - Executing accuracy evaluation method
2025-04-14 21:40:11,244 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
{
  "metrics": {
    "Accuracy": {
      "name": "Accuracy",
      "acronym": "Acc",
      "description": "Coming soon!",
      "status": "OK",
      "results": {
        "test_cases": [{
          "name": "EXIST2025",
          "average": 0.6557377049180327
        }],
        "average_per_test_case": 0.6557377049180327
      }
    },
    "FMeasure": {
      "name": "F-Measure",
      "acronym": "F1",
      "description": "Coming soon!",
      "status": "OK",
      "results": {
        "test_cases": [{
          "name": "EXIST2025",
          "classes": {
            "YES": 0.72,
            "NO": 0.5531914893617

In [33]:
from sklearn.model_selection import train_test_split

# Use train_test_split to split training data into training and validation sets
train_text_en, val_text_en, train_labels_en, val_labels_en, train_id_en, val_id_en = train_test_split(train_df_en["text"].to_numpy(),
                                                                  train_df_en["labels_task3_1"].to_numpy(),
                                                                  train_df_en["id_EXIST"].to_numpy(),
                                                                  test_size=0.1, # dedicate 10% of samples to validation set
                                                                  random_state=42) # random state for reproducibility

In [34]:
len(train_text_en), len(train_labels_en), len(val_text_en), len(val_labels_en)

(714, 714, 80, 80)

In [35]:
# Find average number of tokens (words) in training Tweets
round(sum([len(i.split()) for i in train_text_en])/len(train_text_en))

60

In [36]:
# Setup text vectorization with custom variables
max_vocab_length = 10000 # max number of words to have in our vocabulary
max_length = 60 # max length our sequences will be (e.g. how many words from a Tweet does our model see?)

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [37]:
# Fit the text vectorizer to the training text
text_vectorizer.adapt(train_text_en)

In [38]:
tf.random.set_seed(42)
from tensorflow.keras import layers

embedding = layers.Embedding(input_dim=max_vocab_length, # set input shape
                             output_dim=128, # set size of embedding vector
                             embeddings_initializer="uniform", # default, intialize randomly
                             input_length=max_length, # how long is each input
                             name="embedding_1")
embedding

/Users/naranja/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


<Embedding name=embedding_1, built=False>

In [39]:
# Get a random sentence from training set
result = random.choice(train_text_en)
print(f"Original text:\n{result}\
      \n\nEmbedded version:")

# Embed the random sentence (turn it into numerical representation)
sample_embed = embedding(text_vectorizer([result]))
sample_embed

Original text:
like video follow clean previous detected text removing repetitions leaving text brief essential clean completely readable english information      

Embedded version:


<tf.Tensor: shape=(1, 60, 128), dtype=float32, numpy=
array([[[-0.03632088,  0.03324653,  0.02934397, ..., -0.00689464,
          0.04026189, -0.03794586],
        [-0.01442559, -0.03705555, -0.04466436, ...,  0.02832781,
          0.00755767,  0.02249042],
        [ 0.01820645, -0.02528073,  0.01969485, ..., -0.00542873,
         -0.01779107,  0.03611464],
        ...,
        [-0.03048091,  0.04604176,  0.00263858, ...,  0.02840415,
         -0.00813578,  0.03265736],
        [-0.03048091,  0.04604176,  0.00263858, ...,  0.02840415,
         -0.00813578,  0.03265736],
        [-0.03048091,  0.04604176,  0.00263858, ...,  0.02840415,
         -0.00813578,  0.03265736]]], dtype=float32)>

In [40]:
# Model 0: Getting a baseline - Naive Bayes
train_labels_en = train_labels_en.astype('int') # sklearn cannot recognize object type
val_labels_en = val_labels_en.astype('int')

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Create tokenization and modelling pipeline
model_0 = Pipeline([
                    ("tfidf", TfidfVectorizer()), # convert words to numbers using tfidf
                    ("clf", MultinomialNB()) # model the text
])

# Fit the pipeline to the training data
model_0.fit(train_text_en, train_labels_en)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [42]:
baseline_score = model_0.score(val_text_en, val_labels_en)
print(f"Our baseline model achieves an accuracy of: {baseline_score*100:.2f}%")

Our baseline model achieves an accuracy of: 62.50%


In [43]:
# Make predictions
baseline_preds = model_0.predict(val_text_en)
baseline_preds[:20]

array([1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0])

In [44]:
# Prediction on test dataset
test_sentences = test_df["text"].to_list()
for test_sample in test_sentences:
    pred_prob = tf.squeeze(model_0.predict([test_sample])) # has to be list
    pred = tf.round(pred_prob)
    print(f"Pred: {int(pred)}")

Pred: 0
Pred: 0
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 1


In [45]:
baseline_preds = list(map(lambda x: "YES" if x == 1 else "NO", baseline_preds))
baseline_preds = np.array(baseline_preds)
val_labels_arr = list(map(lambda x: "YES" if x == 1 else "NO", val_labels_en))
val_labels_arr = np.array(val_labels_arr)

In [46]:
import pandas as p

preds_dict = {
    'test_case': ['EXIST2025']*len(val_text_en),
    'id': val_id_en,
    'value': baseline_preds,
}

preds_dict = pd.DataFrame(preds_dict)
output_path = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/predictions_en.json'
with open(output_path, 'w', encoding='utf-8') as output_file:
       output_file.write(preds_dict.to_json(orient='records'))

In [47]:
val_dict = {
    'test_case': ['EXIST2025']*len(val_text_en),
    'id': val_id_en,
    'value': val_labels_arr,
}

val_dict = pd.DataFrame(val_dict)
output_path = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/goldgroundtruth_en.json'
with open(output_path, 'w', encoding='utf-8') as output_file:
       output_file.write(val_dict.to_json(orient='records'))

In [48]:
from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils
from pyevall.metrics.metricfactory import MetricFactory

test = PyEvALLEvaluation()
preds = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/predictions_en.json'
labels = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/goldgroundtruth_en.json'
metrics = [
    MetricFactory.Accuracy.value,
    MetricFactory.FMeasure.value,
]
params= dict()
report = test.evaluate(preds, labels, metrics, **params) 
report.print_report()

2025-04-14 21:40:12,214 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['Accuracy', 'FMeasure']
2025-04-14 21:40:12,249 - pyevall.metrics.metrics - INFO -             evaluate() - Executing accuracy evaluation method
2025-04-14 21:40:12,282 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
{
  "metrics": {
    "Accuracy": {
      "name": "Accuracy",
      "acronym": "Acc",
      "description": "Coming soon!",
      "status": "OK",
      "results": {
        "test_cases": [{
          "name": "EXIST2025",
          "average": 0.625
        }],
        "average_per_test_case": 0.625
      }
    },
    "FMeasure": {
      "name": "F-Measure",
      "acronym": "F1",
      "description": "Coming soon!",
      "status": "OK",
      "results": {
        "test_cases": [{
          "name": "EXIST2025",
          "classes": {
            "NO": 0.6590909090909092,
            "YES": 0.5833333333333333
        

In [49]:
from sklearn.model_selection import train_test_split

# Use train_test_split to split training data into training and validation sets
train_text, val_text, train_labels, val_labels, train_id, val_id = train_test_split(train_df["text"].to_numpy(),
                                                                  train_df["labels_task3_1"].to_numpy(),
                                                                  train_df["id_EXIST"].to_numpy(),
                                                                  test_size=0.1, # dedicate 10% of samples to validation set
                                                                  random_state=42) # random state for reproducibility

In [50]:
len(train_text), len(train_labels), len(val_text), len(val_labels)

(1805, 1805, 201, 201)

In [51]:
# Find average number of tokens (words) in training Tweets
round(sum([len(i.split()) for i in train_text])/len(train_text))

57

In [52]:
# Setup text vectorization with custom variables
max_vocab_length = 10000 # max number of words to have in our vocabulary
max_length = 57 # max length our sequences will be (e.g. how many words from a Tweet does our model see?)

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [53]:
# Fit the text vectorizer to the training text
text_vectorizer.adapt(train_text)

In [54]:
tf.random.set_seed(42)
from tensorflow.keras import layers

embedding = layers.Embedding(input_dim=max_vocab_length, # set input shape
                             output_dim=128, # set size of embedding vector
                             embeddings_initializer="uniform", # default, intialize randomly
                             input_length=max_length, # how long is each input
                             name="embedding_1")
embedding

/Users/naranja/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


<Embedding name=embedding_1, built=False>

In [55]:
# Get a random sentence from training set
result = random.choice(train_text)
print(f"Original text:\n{result}\
      \n\nEmbedded version:")

# Embed the random sentence (turn it into numerical representation)
sample_embed = embedding(text_vectorizer([result]))
sample_embed

Original text:
copien link xfa hombres violan problema serio pretendas importa mierda abuso hacia hombres idiota move girls trying      

Embedded version:


<tf.Tensor: shape=(1, 57, 128), dtype=float32, numpy=
array([[[-3.2109991e-02, -3.1650174e-02,  3.3580076e-02, ...,
         -1.1256240e-02, -1.4946401e-02,  3.3562969e-02],
        [ 2.7960729e-02, -3.1121194e-02, -7.7556856e-03, ...,
          4.7124512e-03, -9.2163682e-06, -3.0880606e-02],
        [-3.2109991e-02, -3.1650174e-02,  3.3580076e-02, ...,
         -1.1256240e-02, -1.4946401e-02,  3.3562969e-02],
        ...,
        [ 9.0460069e-03,  3.1145226e-02,  1.2384713e-02, ...,
         -2.3595965e-02, -4.6066295e-02, -4.7684502e-02],
        [ 9.0460069e-03,  3.1145226e-02,  1.2384713e-02, ...,
         -2.3595965e-02, -4.6066295e-02, -4.7684502e-02],
        [ 9.0460069e-03,  3.1145226e-02,  1.2384713e-02, ...,
         -2.3595965e-02, -4.6066295e-02, -4.7684502e-02]]], dtype=float32)>

In [56]:
# Model 0: Getting a baseline - Naive Bayes
train_labels = train_labels.astype('int') # sklearn cannot recognize object type
val_labels = val_labels.astype('int')

In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Create tokenization and modelling pipeline
model_0 = Pipeline([
                    ("tfidf", TfidfVectorizer()), # convert words to numbers using tfidf
                    ("clf", MultinomialNB()) # model the text
])

# Fit the pipeline to the training data
model_0.fit(train_text, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [58]:
baseline_score = model_0.score(val_text, val_labels)
print(f"Our baseline model achieves an accuracy of: {baseline_score*100:.2f}%")

Our baseline model achieves an accuracy of: 62.69%


In [59]:
# Make predictions
baseline_preds = model_0.predict(val_text)
baseline_preds[:20]

array([0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0])

In [60]:
# Prediction on test dataset
test_sentences = test_df["text"].to_list()
for test_sample in test_sentences:
    pred_prob = tf.squeeze(model_0.predict([test_sample])) # has to be list
    pred = tf.round(pred_prob)
    print(f"Pred: {int(pred)}")

Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 1
Pred: 1
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 0
Pred: 0


In [61]:
baseline_preds = list(map(lambda x: "YES" if x == 1 else "NO", baseline_preds))
baseline_preds = np.array(baseline_preds)
val_labels_arr = list(map(lambda x: "YES" if x == 1 else "NO", val_labels))
val_labels_arr = np.array(val_labels_arr)

In [62]:
import pandas as p

preds_dict = {
    'test_case': ['EXIST2025']*len(val_text),
    'id': val_id,
    'value': baseline_preds,
}

preds_dict = pd.DataFrame(preds_dict)
output_path = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/predictions_en_es.json'
with open(output_path, 'w', encoding='utf-8') as output_file:
       output_file.write(preds_dict.to_json(orient='records'))

In [63]:
val_dict = {
    'test_case': ['EXIST2025']*len(val_text),
    'id': val_id,
    'value': val_labels_arr,
}

val_dict = pd.DataFrame(val_dict)
output_path = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/goldgroundtruth_en_es.json'
with open(output_path, 'w', encoding='utf-8') as output_file:
       output_file.write(val_dict.to_json(orient='records'))

In [64]:
from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils
from pyevall.metrics.metricfactory import MetricFactory

test = PyEvALLEvaluation()
preds = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/predictions_en_es.json'
labels = '/Users/naranja/Downloads/NLP Course/lab3_materials/dataset_task3_exist2025/goldgroundtruth_en_es.json'
metrics = [
    MetricFactory.Accuracy.value,
    MetricFactory.FMeasure.value,
]
params= dict()
report = test.evaluate(preds, labels, metrics, **params) 
report.print_report()

2025-04-14 21:40:13,468 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['Accuracy', 'FMeasure']
2025-04-14 21:40:13,540 - pyevall.metrics.metrics - INFO -             evaluate() - Executing accuracy evaluation method
2025-04-14 21:40:13,623 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
{
  "metrics": {
    "Accuracy": {
      "name": "Accuracy",
      "acronym": "Acc",
      "description": "Coming soon!",
      "status": "OK",
      "results": {
        "test_cases": [{
          "name": "EXIST2025",
          "average": 0.6268656716417911
        }],
        "average_per_test_case": 0.6268656716417911
      }
    },
    "FMeasure": {
      "name": "F-Measure",
      "acronym": "F1",
      "description": "Coming soon!",
      "status": "OK",
      "results": {
        "test_cases": [{
          "name": "EXIST2025",
          "classes": {
            "NO": 0.6445497630331755,
            "YES": 0